# pg_ducklake: a lakehouse table inside Postgres

[DuckLake](https://ducklake.select) is a table format from the DuckDB team: the data is Parquet
files, and the catalog (which files make up a table, their statistics, every snapshot) is ordinary
tables in a SQL database. [pg_ducklake](https://github.com/relytcloud/pg_ducklake) puts both ends
in Postgres. `CREATE TABLE ... USING ducklake` makes a table whose rows are written to Parquet,
whose catalog lives in the `ducklake` schema of the same database, and whose queries DuckDB runs,
columnar and vectorised, inside the Postgres backend.

So one database can hold row tables for the application and column tables for analytics, and a
query can join the two. This notebook measures what that buys and walks through what a lake table
adds: snapshots, time travel, a change feed and partitions.

Everything runs as plain SQL through psycopg on tables named `nb_*`, which are dropped at the end.

**Prerequisites**: the ducklake server is up (`just db ducklake`, port 5443). It runs the upstream
image `pgducklake/pgducklake:18-v1.0.2`, Postgres 18 with the extension preloaded.

In [ ]:
# Parameters. Papermill overrides these with `-p NAME value`.
SERVER = "ducklake"  # a server in src/settings.py with pg_ducklake installed
N_ROWS = 5_000_000  # rows in the events table
RUNS = 5  # timed runs per query; the median is reported

In [ ]:
import pathlib
import statistics
import sys
import time

from dotenv import load_dotenv

ROOT = next(
    p
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "src" / "db.py").exists()
)
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

import pandas as pd
import psycopg

from settings import settings

DSN = settings.sqlalchemy_url(SERVER).replace("postgresql+psycopg://", "postgresql://")
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

conn = psycopg.connect(DSN, autocommit=True)


def q(sql: str, *args) -> pd.DataFrame:
    """Run one statement and show what came back."""
    cur = conn.execute(sql, args or None)
    if cur.description is None:
        return pd.DataFrame()
    return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


def timed(sql: str, runs: int = RUNS) -> float:
    """Median wall time of `sql` in milliseconds, after one warm-up run."""
    conn.execute(sql).fetchall()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        conn.execute(sql).fetchall()
        times.append((time.perf_counter() - t0) * 1000)
    return statistics.median(times)


def snapshot_now() -> int:
    """The id of the latest snapshot."""
    return int(conn.execute("SELECT * FROM ducklake.current_snapshot()").fetchone()[0])


def files_view(files: pd.DataFrame) -> pd.DataFrame:
    """list_files output with the paths cut to file names and the sizes in kB."""
    out = pd.DataFrame({"data_file": files.data_file.str.replace(r".*/", "", regex=True),
                        "data kB": (files.data_file_size_bytes / 1024).round()})
    if files.delete_file.notna().any():
        out["delete_file"] = files.delete_file.str.replace(r".*/", "", regex=True)
        out["delete kB"] = (files.delete_file_size_bytes / 1024).round(1)
    return out


def drop_nb_tables() -> None:
    for (name,) in conn.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'public' AND tablename LIKE 'nb%'").fetchall():
        conn.execute(f"DROP TABLE {name}")


drop_nb_tables()
q("""
SELECT current_setting('server_version') AS postgres,
       (SELECT extversion FROM pg_extension WHERE extname = 'pg_ducklake') AS pg_ducklake,
       current_setting('shared_preload_libraries') AS preloaded
""")

## 1. The same table twice

`nb_events_heap` is an ordinary Postgres table with a primary key: an id, a timestamp spread over
30 days, a device, a user and an amount. `nb_events_lake` is a copy made with `USING ducklake`,
which writes the rows out as Parquet in one go.

In [ ]:
t0 = time.perf_counter()
conn.execute(f"""
CREATE TABLE nb_events_heap AS
SELECT i AS id,
       timestamptz '2026-09-01' + (i % (30 * 86400)) * interval '1 second' AS ts,
       (ARRAY['web', 'ios', 'android'])[1 + i % 3] AS device,
       ((i::bigint * 7919) % 100000)::int AS user_id,
       round((random() * 100)::numeric, 2) AS amount
FROM generate_series(1, {N_ROWS}) AS i
""")
conn.execute("ALTER TABLE nb_events_heap ADD PRIMARY KEY (id)")
conn.execute("VACUUM ANALYZE nb_events_heap")
heap_s = time.perf_counter() - t0

# the catalog keeps the history of earlier runs too; later cells show only what this run did
first_snapshot = snapshot_now() + 1
t0 = time.perf_counter()
conn.execute("CREATE TABLE nb_events_lake USING ducklake AS SELECT * FROM nb_events_heap")
lake_s = time.perf_counter() - t0

files = q("SELECT * FROM ducklake.list_files('nb_events_lake')")
sizes = q("""
SELECT pg_size_pretty(pg_table_size('nb_events_heap')) AS heap_table,
       pg_size_pretty(pg_indexes_size('nb_events_heap')) AS heap_index
""")
sizes["lake_parquet"] = f"{files.data_file_size_bytes.sum() / 2**20:.0f} MB in {len(files)} file(s)"
print(f"heap: built, keyed and vacuumed in {heap_s:.1f} s; lake: copied from the heap in {lake_s:.1f} s")
sizes

## 2. Analytics: columns win

Three analytical queries, each run on both tables. They read a few columns of every row, which is
what Parquet is laid out for: DuckDB reads only those columns, compressed, in vectors of 2,048
values. The heap reads whole rows, 8 kB page by page, though Postgres does use parallel workers.

In [ ]:
QUERIES = {
    "sum by device": "SELECT device, count(*), sum(amount) FROM {t} GROUP BY device",
    "daily distinct users": "SELECT date_trunc('day', ts) AS day, count(DISTINCT user_id) FROM {t} GROUP BY 1",
    "top users by spend": "SELECT user_id, sum(amount) AS spend FROM {t} GROUP BY 1 ORDER BY 2 DESC LIMIT 10",
}
rows = []
for name, sql in QUERIES.items():
    heap_ms = timed(sql.format(t="nb_events_heap"))
    lake_ms = timed(sql.format(t="nb_events_lake"))
    rows.append({"query": name, "heap ms": round(heap_ms, 1), "ducklake ms": round(lake_ms, 1), "speed-up": f"{heap_ms / lake_ms:.1f}x"})
analytics = pd.DataFrame(rows)
analytics

In [ ]:
# both tables give the same answer
q("""
SELECT 'heap' AS t, device, count(*), sum(amount) FROM nb_events_heap GROUP BY device
UNION ALL
SELECT 'lake', device, count(*), sum(amount) FROM nb_events_lake GROUP BY device
ORDER BY device, t
""")

## 3. Point lookups: rows win

The other side of the trade. Fetching one row by id is a B-tree descent on the heap, a few pages.
A lake table has no index: DuckDB prunes files and row groups by their min/max statistics, then
scans what is left.

In [ ]:
lookup = pd.DataFrame([
    {"table": t, "one row by id, ms": round(timed(f"SELECT * FROM {t} WHERE id = 1234567", runs=20), 2)}
    for t in ["nb_events_heap", "nb_events_lake"]
])
lookup

## 4. Where the table lives

The catalog is Postgres tables in the `ducklake` schema: one row per table, per column, per data
file, per snapshot. The data is Parquet files, by default under the Postgres data directory
(`ducklake.default_table_path` can point them at S3, GCS, R2 or Azure instead).

In [ ]:
q("""
SELECT t.table_name, f.path, f.record_count, pg_size_pretty(f.file_size_bytes) AS size
FROM ducklake.ducklake_data_file f
JOIN ducklake.ducklake_table t USING (table_id)
WHERE t.table_name = 'nb_events_lake' AND t.end_snapshot IS NULL AND f.end_snapshot IS NULL
""")

In [ ]:
q("SELECT option_name, value FROM ducklake.options()")

## 5. Writes: inlining, delete files, compaction

Parquet files are immutable, so small writes would mean many tiny files. DuckLake has two answers.
An insert below a row limit is **inlined**: the rows go into a Postgres table in the catalog, and
queries read them alongside the Parquet until a flush writes them out. An update or delete writes
a small **delete file** naming the row positions that no longer count, next to the untouched data
file.

In [ ]:
conn.execute("INSERT INTO nb_events_lake VALUES (9000001, now(), 'web', 1, 9.99)")
print("data files after a one-row insert:", len(q("SELECT * FROM ducklake.list_files('nb_events_lake')")))
q("""
SELECT i.table_name AS inlined_into
FROM ducklake.ducklake_inlined_data_tables i
JOIN ducklake.ducklake_table t USING (table_id)
WHERE t.table_name = 'nb_events_lake' AND t.end_snapshot IS NULL
""")

In [ ]:
# the inlined row is visible to queries before any Parquet exists for it
q("SELECT * FROM nb_events_lake WHERE id = 9000001")

In [ ]:
conn.execute("SELECT * FROM ducklake.flush_inlined_data('nb_events_lake'::regclass)")
before_update = snapshot_now()
conn.execute("UPDATE nb_events_lake SET amount = 0 WHERE id <= 1000")
conn.execute("DELETE FROM nb_events_lake WHERE device = 'ios' AND user_id < 100")
files_view(q("SELECT * FROM ducklake.list_files('nb_events_lake')"))

## 6. Snapshots and time travel

Every commit is a snapshot, recorded in the catalog with what it changed. A query can read the
table as of any snapshot that has not been expired, which makes "what did this table look like
before the batch job" a function call.

In [ ]:
# snapshots() returns DuckLake rows, so take them whole and pick the columns in pandas
snaps = q("SELECT * FROM ducklake.snapshots()")[["snapshot_id", "snapshot_time", "changes"]]
snaps[snaps.snapshot_id >= first_snapshot]

In [ ]:
# the snapshot recorded just before the UPDATE and DELETE of section 5
q(f"""
SELECT 'now' AS version, count(*) AS rows, sum(amount) AS total FROM nb_events_lake
UNION ALL
SELECT 'snapshot {before_update}', count(*), sum(t['amount']::numeric) FROM ducklake.time_travel('nb_events_lake'::regclass, {before_update}) t
""")

## 7. A change feed

`table_changes(table, from, to)` returns the rows inserted, updated and deleted between two
snapshots, which is what an incremental job downstream needs. Here it reads the inserts made by
two new commits.

In pg_ducklake 1.0.2 a range that contains a delete fails with `function struct_pack(...) does not
exist`, and the connection's DuckDB loses its catalog until it reconnects, so this cell stays on a
range of inserts.

In [ ]:
start = snapshot_now() + 1
conn.execute("INSERT INTO nb_events_lake VALUES (9000002, now(), 'ios', 2, 5.00)")
conn.execute("INSERT INTO nb_events_lake VALUES (9000003, now(), 'android', 3, 7.50)")
end = snapshot_now()
q(f"SELECT * FROM ducklake.table_changes('nb_events_lake'::regclass, {start}, {end})")

## 8. Partitions

`set_partition` makes later writes split their files by a column (or a transform of one, such as
`year(ts)`), and a filter on that column then skips whole files.

In [ ]:
conn.execute("CREATE TABLE nb_events_part (id bigint, ts timestamptz, device text, amount numeric(10, 2)) USING ducklake")
conn.execute("CALL ducklake.set_partition('nb_events_part'::regclass, 'device')")
conn.execute("INSERT INTO nb_events_part SELECT id, ts, device, amount FROM nb_events_heap")
part = q("SELECT * FROM ducklake.list_files('nb_events_part')")
pd.DataFrame({"file": part.data_file.str.replace(r".*/nb_events_part/", "", regex=True),
              "MB": (part.data_file_size_bytes / 2**20).round(1)}).sort_values("file")

In [ ]:
pd.DataFrame([
    {"table": t, "sum for one device, ms": round(timed(f"SELECT sum(amount) FROM {t} WHERE device = 'android'"), 1)}
    for t in ["nb_events_heap", "nb_events_lake", "nb_events_part"]
])

On a local disk, at five million rows, the partitioned table is no faster than the single file:
DuckDB already reads one compressed column of a 78 MB file in a few tens of milliseconds, and the
row-group statistics inside it do part of a partition's job. Partitions pay off with object
storage, where a skipped file is a skipped download, and with tables of many files.

## 9. Joining rows and columns

A lake table joins any Postgres table. DuckDB plans the whole query and reads the heap side
through its Postgres scan, so the application's `users` can label the lake's events in one
statement.

In [ ]:
conn.execute("CREATE TABLE nb_users AS SELECT u AS user_id, (u % 5 = 0) AS pro FROM generate_series(0, 99999) AS u")
q("""
SELECT u.pro, count(*) AS events, round(sum(e.amount), 2) AS spend
FROM nb_events_lake e JOIN nb_users u USING (user_id)
GROUP BY u.pro ORDER BY u.pro
""")

In [ ]:
plan = q("EXPLAIN SELECT u.pro, count(*) FROM nb_events_lake e JOIN nb_users u USING (user_id) GROUP BY u.pro")
print("\n".join(plan.iloc[:, 0]))

## Cleanup

In [ ]:
drop_nb_tables()
conn.close()
print("clean")